<h1>Capítulo 8 - Busca Semântica e Geração Aumentada por Recuperação (RAG)</h1>
<i>Explorando uma parte essencial das aplicações com LLMs: a busca.</i>

**Disciplina:** Tópicos Especiais em Inteligência Artificial (IFPE, Campus Paulista). **Aula:** Busca Semântica.

---

## O que vamos construir nesta aula

Ao longo do notebook, vamos montar, passo a passo, as peças de um sistema de busca e de um sistema RAG, na mesma ordem dos slides:

| Slide da aula | Seção do notebook | O que você vai ver na prática |
|---|---|---|
| Recuperação densa; embeddings como pontos no espaço | Exemplo de recuperação densa | transformar frases em vetores e buscar os mais próximos da pergunta |
| Dividir textos longos em blocos | 1. Obtendo o texto e dividindo em blocos | cada frase do texto vira um bloco (*chunk*) |
| Base de conhecimento para um banco vetorial | 2 e 3. Embeddings e índice | gerar os vetores e guardá-los em um índice FAISS |
| Busca pelo vizinho mais próximo | 4. Buscando no índice | encontrar os blocos mais próximos de uma pergunta |
| (comparação) | Busca por palavras-chave (BM25) | por que a busca tradicional falha quando as palavras não coincidem |
| Limitações e ajuste fino | Limitações da recuperação densa | a busca densa sempre devolve algo, mesmo quando não há resposta |
| Reranking | Exemplo de reranking | reordenar os candidatos com um modelo mais preciso |
| Retrieval-Augmented Generation | Geração aumentada por recuperação | responder com um LLM usando apenas os trechos recuperados, com citações |

O texto usado como base de conhecimento é a introdução do artigo da Wikipédia sobre o filme *Interstellar* (em inglês), o mesmo exemplo do livro e dos slides.

## Nota de adaptação: versão com a API da NVIDIA

Esta versão substitui as chamadas à API da Cohere, usada no livro, por modelos hospedados no catálogo gratuito da NVIDIA (build.nvidia.com). Os conceitos do Capítulo 8 são os mesmos; muda apenas o provedor.

**Os modelos da API podem ser desativados.** Em agosto de 2026, a NVIDIA retirou do catálogo  modelos, e as chamadas passaram a devolver o erro **410 Gone**. Por isso, todos os nomes de modelos estão concentrados na célula de configuração (seção 1). Se receber um erro 410, procure um substituto em https://build.nvidia.com/models e altere apenas essa célula.

**Como obter a chave.** Crie uma conta gratuita em https://build.nvidia.com (NVIDIA Developer Program, sem cartão de crédito) e gere uma chave em *Get API Key*. A chave começa com `nvapi-` e é exibida uma única vez. Cada estudante deve usar a própria chave.

**Limites.** O nível gratuito limita o número de requisições por minuto, por modelo. Uma execução completa deste notebook faz menos de 10 chamadas, bem abaixo do limite.

**Hardware.** Toda a inferência ocorre na nuvem da NVIDIA; o notebook roda em CPU (Colab sem GPU ou uma instância EC2 simples).

**Diferenças em relação ao livro.**

1. As distâncias e as pontuações de relevância serão diferentes das impressas no livro, pois os modelos são outros. O reranker da NVIDIA devolve *logits* (valores sem limite fixo, que podem ser negativos), e não o `relevance_score` entre 0 e 1 da Cohere. O que importa é a **ordem** dos resultados.
2. A API da Cohere devolve citações nativas (`response.citations`). Aqui as citações são pedidas no próprio prompt, no formato `[n]`, e extraídas com expressão regular. É uma aproximação: o modelo pode citar errado ou esquecer de citar.
3. Os textos enviados saem da sua máquina e são processados na nuvem da NVIDIA. **Não envie dados pessoais reais** (LGPD).

**Referências.** ALAMMAR, J.; GROOTENDORST, M. *Hands-On Large Language Models*. O'Reilly, 2024, Cap. 8. Catálogo de modelos da NVIDIA: https://build.nvidia.com/models

### Instalação dos pacotes

A célula abaixo instala apenas o necessário para esta aula (`faiss-cpu`, `langchain-community` e `rank_bm25`) e **é a que você deve executar no Colab**. Os demais pacotes (`openai`, `requests`, `pandas`, `scikit-learn`, `tqdm`) já vêm instalados no Colab.

**Observação.** Durante a instalação, o Colab pode exibir um aviso de conflito de dependências envolvendo o pacote `requests` e o `google-colab`. É apenas um aviso: a instalação foi concluída e o notebook funciona normalmente.

**Não é necessário usar GPU.** Diferentemente do notebook original, aqui os modelos rodam na nuvem da NVIDIA.

In [47]:
!pip install -q faiss-cpu langchain-community rank_bm25

# Exemplo de recuperação densa (*dense retrieval*)

Sistemas de busca tradicionais procuram documentos que contenham as **mesmas palavras** da consulta. A **recuperação densa** procura documentos com o mesmo **significado**.

A ideia central são os **embeddings**: um modelo transforma cada texto em um vetor de números (neste notebook, 2048 números por texto). Esses vetores podem ser vistos como pontos em um espaço de muitas dimensões, organizados de forma que **textos com significados parecidos fiquem próximos**. Para buscar, transformamos também a pergunta em um vetor e procuramos os textos cujos pontos estão mais perto dela.

O fluxo completo, como nos slides "Base de conhecimento para um banco de dados vetorial" e "Busca semântica com modelos de linguagem", tem quatro etapas, que correspondem às seções a seguir:

1. **Dividir** os documentos em blocos (*chunks*).
2. **Gerar o embedding** de cada bloco.
3. **Guardar** os vetores em um índice (o "banco de dados vetorial").
4. **Buscar**: gerar o embedding da pergunta e recuperar os blocos mais próximos.


## 1. Obtendo o texto e dividindo em blocos

### 1.1 Configuração da chave e dos modelos

A primeira célula lê a chave da API e define, em um único lugar, os endereços e os nomes dos modelos usados no restante do notebook.

**Antes de executar:** no Colab, clique no ícone de chave (*Secrets*) na barra lateral esquerda, crie um segredo chamado `NVIDIA_API_KEY` com a sua chave e ative o acesso para este notebook. Assim a chave não fica escrita no código, e você não corre o risco de compartilhá-la junto com o notebook.

O que a célula faz:

* `NVIDIA_BASE_URL` é o endereço da API da NVIDIA, compatível com a API da OpenAI. Por isso podemos usar a biblioteca `openai` (o objeto `client`), trocando apenas o endereço e a chave. Esse padrão é comum: muitos provedores e servidores locais, como o `llama-server`, seguem a mesma interface.
* `EMBED_MODEL` gera os embeddings; `RERANK_MODEL` e `RERANK_URL` são usados no reranking; `CHAT_MODEL` é o LLM que gera a resposta final do RAG.

In [48]:
import os
import getpass
from openai import OpenAI
from google.colab import userdata

# Cole sua chave da NVIDIA quando solicitado (formato nvapi-...). Nunca publique a chave.
try:
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
except Exception:
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key: ")

# Endpoints e modelos (confira o catálogo em https://build.nvidia.com/models antes da aula)
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
EMBED_MODEL = "nvidia/nemotron-3-embed-1b"
RERANK_MODEL = "nvidia/llama-nemotron-rerank-vl-1b-v2"
RERANK_URL = f"https://ai.api.nvidia.com/v1/retrieval/{RERANK_MODEL}/reranking"
CHAT_MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

# Cliente compatível com a API da OpenAI apontando para a NVIDIA
client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=os.environ["NVIDIA_API_KEY"])

### 1.2 O texto e a divisão em blocos

O texto é a introdução do artigo da Wikipédia sobre *Interstellar*. Para indexá-lo, precisamos decidir **o que vira um vetor**. Os slides apresentam duas opções:

* **um vetor por documento**: simples, mas um único vetor precisa resumir o texto inteiro, e muitos detalhes ficam "diluídos" e difíceis de encontrar;
* **vários vetores por documento**: o texto é dividido em blocos menores, e cada bloco recebe o próprio embedding. O índice passa a ser formado pelos blocos, e não pelos documentos inteiros.

Aqui usamos a segunda opção com a estratégia mais simples: **cada frase é um bloco**. O código divide o texto a cada ponto final (`text.split('.')`) e remove espaços e quebras de linha das bordas. O resultado, `texts`, é a lista de frases que será indexada.

Essa divisão ingênua tem limitações. Um ponto também aparece em abreviações ("Dr.") e em números ("2.5"), que seriam cortados no meio. Além disso, uma frase isolada pode perder o contexto: "It stars Matthew McConaughey..." não diz de qual filme está falando. Os slides mostram alternativas: dividir por parágrafo, por número de tokens ou usar **janelas sobrepostas** (*overlap*), em que blocos vizinhos compartilham parte do texto para não perder o contexto nas bordas.

In [49]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

## 2. Gerando os embeddings dos blocos

A função `embed` envia os textos ao modelo de embeddings da NVIDIA, em lotes de até 32 textos por requisição, e devolve uma matriz NumPy com um vetor por texto.

Um detalhe importante é o parâmetro `input_type`. Modelos de embedding para busca são **assimétricos**: perguntas costumam ser curtas ("how precise was the science"), enquanto os documentos são afirmações mais longas. Por isso, o modelo codifica de forma diferente:

* `"passage"`: os blocos que serão indexados (o livro, com a Cohere, chama isso de `search_document`);
* `"query"`: a pergunta do usuário (`search_query`, no livro).

Trocar os dois não gera erro, mas piora a qualidade da busca sem nenhum aviso.

A saída `(15, 2048)` significa **15 blocos** (as frases do texto), cada um representado por um vetor de **2048 dimensões**.

In [50]:
import numpy as np

def embed(texts, input_type, batch_size=32):
    # Equivalente ao co.embed: "search_document" vira "passage" e "search_query" vira "query"
    nim_type = "query" if input_type == "search_query" else "passage"
    vectors = []
    for i in range(0, len(texts), batch_size):
        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=texts[i:i + batch_size],
            encoding_format="float",
            extra_body={"input_type": nim_type, "truncate": "NONE"},
        )
        vectors.extend(item.embedding for item in response.data)
    return np.array(vectors)

# Get the embeddings
embeds = embed(texts, input_type="search_document")
print(embeds.shape)

(15, 2048)


## 3. Construindo o índice de busca

Com os vetores prontos, precisamos de uma estrutura que encontre rapidamente os vizinhos mais próximos de um novo vetor. Usamos o **FAISS**, biblioteca de busca vetorial da Meta:

* `faiss.IndexFlatL2(dim)` cria um índice que mede a **distância euclidiana (L2)** entre vetores de dimensão `dim` (aqui, 2048);
* `index.add(...)` insere os 15 vetores no índice. O FAISS exige `float32`, por isso a conversão com `np.float32`.

O índice "Flat" faz uma **busca exata**: compara a pergunta com todos os vetores, um a um. É a "maneira mais direta" citada no slide "Busca pelo vizinho mais próximo" e funciona muito bem para alguns milhares de vetores. Com milhões de vetores, usam-se índices **aproximados** (como HNSW ou IVF), que trocam um pouco de precisão por muita velocidade. Os **bancos de dados vetoriais** (Chroma, Qdrant, Milvus, pgvector, entre outros) oferecem esses índices junto com armazenamento persistente, filtros e atualização dos dados.

**Distância ou similaridade?** Aqui, quanto **menor** a distância L2, mais parecidos os textos. Outra opção comum é a **similaridade de cosseno**, em que **maior** significa mais parecido. No FAISS, ela é obtida com `IndexFlatIP` após normalizar os vetores com `faiss.normalize_L2`.

In [51]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))

## 4. Buscando no índice

A função `search` implementa a etapa de consulta do fluxo:

1. gera o embedding da pergunta, com `input_type="search_query"`;
2. pede ao índice os `number_of_results` vizinhos mais próximos. O FAISS devolve as distâncias e as posições desses vetores;
3. usa as posições para recuperar as frases originais em `texts` e monta uma tabela com o texto e a distância;
4. devolve a tabela, ordenada da menor para a maior distância.

In [52]:
import pandas as pd

def search(query, number_of_results=3):

  # 1. Get the query's embedding
  query_embed = embed([query], input_type="search_query")[0]

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results

Vamos perguntar "quão precisa era a ciência do filme" (*how precise was the science*). Observe que a frase mais próxima fala de **"scientific accuracy"** e de **elogios de astrônomos**, embora não contenha as palavras "precise" nem "science". A busca encontrou o trecho pelo **significado**, e não pela coincidência de palavras. Lembre-se: na coluna `distance`, valores **menores** indicam maior proximidade.

In [53]:
query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.481185
1,Caltech theoretical physicist and 2017 Nobel l...,1.655933
2,"It received acclaim for its performances, dire...",1.706033


## Comparação: busca por palavras-chave (BM25)

Para entender a vantagem da busca densa, vamos compará-la com a **busca lexical** (por palavras-chave), a abordagem clássica dos mecanismos de busca. Usamos o **BM25**, algoritmo que dá pontuação alta a documentos que contêm as palavras da consulta, valorizando mais as palavras raras na coleção e ajustando pelo tamanho do documento.

A função `bm25_tokenizer` prepara cada texto: converte para minúsculas, separa por espaços, remove a pontuação das bordas e descarta as *stop words* (palavras muito frequentes e pouco informativas, como "the" e "of"). A lista de *stop words* usada é a do inglês, pois o texto está em inglês.

In [54]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

A célula abaixo tokeniza as 15 frases e cria o índice BM25. Diferentemente da busca densa, aqui não há modelo nem API: o índice é apenas uma estatística das palavras de cada frase.

In [55]:
from tqdm import tqdm

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 15/15 [00:00<00:00, 50696.66it/s]


A função `keyword_search` calcula a pontuação BM25 de todas as frases para a consulta, separa as `num_candidates` melhores e mostra as `top_k` primeiras.

In [56]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))


Execute a mesma pergunta usada na busca densa e compare os resultados.

In [57]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


**Como interpretar.** O BM25 coloca em primeiro lugar a frase de abertura ("Interstellar is a 2014 epic **science** fiction film..."), apenas porque ela contém a palavra "science", embora não diga nada sobre a precisão científica do filme. A frase sobre os elogios dos astrônomos, que é a melhor resposta, não aparece, pois não compartilha nenhuma palavra com a consulta. Pontuação **0,000** significa nenhuma palavra em comum: o BM25 está apenas completando a lista.

A busca densa, por outro lado, falha em situações em que o BM25 é forte, como nomes próprios, códigos e termos técnicos exatos (por exemplo, o número de um processo ou de uma disciplina). Por isso, sistemas reais costumam usar **busca híbrida**, combinando as duas abordagens.

## Limitações da recuperação densa

Um índice vetorial **sempre** devolve os vizinhos mais próximos, mesmo quando nenhum deles responde à pergunta. Vamos perguntar algo que o texto não contém: "Qual é a massa da Lua?" (*What is the mass of the moon?*).

In [58]:
query = "What is the mass of the moon?"
results = search(query)
results

Query:'What is the mass of the moon?'
Nearest neighbors:


,texts,distance
0,Set in a dystopian future where humanity is st...,1.807694
1,It has also received praise from many astronom...,1.829117
2,Caltech theoretical physicist and 2017 Nobel l...,1.847440


**Como interpretar.** A busca devolveu três frases sobre o filme, nenhuma delas com a resposta. Compare as distâncias com as da busca anterior: aqui elas são **maiores e muito parecidas entre si**, o que indica que nenhuma frase é realmente próxima da pergunta. Mas o índice não tem como "dizer que não sabe".

Em um sistema RAG, esses trechos irrelevantes seriam enviados ao LLM, que poderia inventar uma resposta a partir deles. Formas de reduzir o problema:

* definir um **limiar de distância**, acima do qual os resultados são descartados (o valor adequado depende do modelo e precisa ser calibrado);
* usar um **reranker** (próxima seção), cuja pontuação mede melhor a relevância;
* instruir o LLM a responder apenas com base nos trechos e a dizer quando eles não contêm a resposta;
* para domínios específicos (por exemplo, normas acadêmicas do IFPE), fazer o **ajuste fino** do modelo de embeddings, que aproxima perguntas e trechos relevantes e afasta os irrelevantes, como mostra o slide "Ajuste fino de modelos de embedding para recuperação densa".

Outras limitações: frases muito curtas ou sem contexto (efeito da divisão em blocos) e respostas que dependem de informações espalhadas por vários blocos.

# Exemplo de reranking

Muitas organizações já têm um sistema de busca. A forma mais simples de incorporar um modelo de linguagem a ele é como uma **segunda etapa**: a busca existente (por palavras-chave, densa ou híbrida) seleciona rapidamente alguns candidatos, e um **reranker** reordena esses candidatos por relevância.

**Por que o reranker é mais preciso?** O modelo de embeddings é um *bi-encoder*: codifica a pergunta e cada documento **separadamente** e depois compara os vetores. O reranker é um *cross-encoder*: lê a **pergunta e o documento juntos** e estima o quanto aquele documento responde àquela pergunta. Isso permite capturar relações mais finas, mas custa caro: é preciso uma passada do modelo para cada par pergunta e documento, e nada pode ser calculado com antecedência. Por isso o reranker é aplicado apenas a poucos candidatos, e não à coleção inteira.

A função `rerank` envia a pergunta e os documentos à API de reranking da NVIDIA e devolve os `top_n` documentos mais relevantes, com a pontuação de cada um (`relevance_score`). Na NVIDIA, essa pontuação é um **logit**: um número sem limite fixo, que pode ser negativo. Ele não é uma probabilidade; o que importa é a **ordem**. Neste primeiro teste, enviamos as 15 frases diretamente ao reranker, sem primeira etapa.

In [59]:
import requests

def rerank(query, documents, top_n=3):
    # Equivalente ao co.rerank: devolve os top_n documentos ordenados por relevância
    response = requests.post(
        RERANK_URL,
        headers={
            "Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}",
            "Accept": "application/json",
        },
        json={
            "model": RERANK_MODEL,
            "query": {"text": query},
            "passages": [{"text": doc} for doc in documents],
            "truncate": "END",
        },
        timeout=60,
    )
    response.raise_for_status()
    rankings = response.json()["rankings"][:top_n]
    return [
        {"index": r["index"], "relevance_score": r["logit"], "text": documents[r["index"]]}
        for r in rankings
    ]

query = "how precise was the science"
results = rerank(query=query, documents=texts, top_n=3)
results

[{'index': 12,
  'relevance_score': -5.453125,
  'text': 'It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'},
 {'index': 13,
  'relevance_score': -6.63671875,
  'text': 'Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time'},
 {'index': 4,
  'relevance_score': -6.66015625,
  'text': 'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar'}]

Mostrando os resultados de forma mais legível: posição, pontuação e texto. A frase sobre a precisão científica aparece em primeiro lugar, com a maior pontuação (o logit menos negativo).

In [60]:
for idx, result in enumerate(results):
    print(idx, result["relevance_score"], result["text"])

0 -5.453125 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 -6.63671875 Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time
2 -6.66015625 Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar


### Busca em duas etapas: BM25 seguido de reranking

Agora combinamos as duas ideias, como no slide "Reranking": o **BM25 seleciona 10 candidatos** (primeira etapa, rápida e barata) e o **reranker reordena** esses candidatos (segunda etapa, mais precisa). A função mostra lado a lado os três primeiros resultados de cada etapa.

In [61]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = rerank(query=query, documents=docs, top_n=top_k)
    for hit in results:
        print("\t{:.3f}\t{}".format(hit["relevance_score"], hit["text"].replace("\n", " ")))

In [62]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	-5.453	It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
	-6.637	Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time
	-6.660	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in bo

**Como interpretar.** O BM25, sozinho, colocou a frase de abertura em primeiro lugar por causa da palavra "science". Entre os 10 candidatos do BM25, porém, estava a frase sobre os elogios dos astrônomos, e o reranker a levou para o primeiro lugar. Esse é o papel da segunda etapa: a primeira etapa precisa apenas **incluir** a resposta entre os candidatos, e o reranker se encarrega de colocá-la no topo.

Note a limitação: se a primeira etapa não incluir o trecho correto entre os candidatos, o reranker não tem como recuperá-lo. Por isso, a primeira etapa costuma devolver dezenas de candidatos, e o reranker fica com os poucos melhores.

# Geração aumentada por recuperação (RAG)

Até aqui, o sistema devolve **trechos** de texto. Um sistema **RAG** (*Retrieval-Augmented Generation*; LEWIS et al., 2020) dá o passo seguinte: entrega os trechos recuperados a um LLM, que redige uma **resposta** à pergunta com base neles. O processo tem duas etapas:

1. **Recuperação** (*retrieval*): buscar os trechos relevantes na base de conhecimento, com busca densa, lexical ou híbrida, opcionalmente seguida de reranking;
2. **Geração fundamentada** (*grounded generation*): montar um novo prompt com a pergunta e os trechos recuperados e pedir ao LLM que responda **apenas com base neles**, indicando as fontes.

Como o modelo responde a partir de documentos fornecidos, e não apenas do que memorizou no treinamento, o RAG **reduz as alucinações** e **melhora a precisão factual**. Ele também permite usar informações que o modelo nunca viu, como regulamentos internos, e mostrar ao usuário de onde veio cada afirmação.

## Exemplo: geração fundamentada com a API

A célula abaixo implementa as duas etapas do RAG para a consulta "income generated" (receita gerada):

1. **Recuperação:** usa a função `search` (busca densa) para obter as 3 frases mais próximas. O comentário do livro lembra que, idealmente, usaríamos busca híbrida.
2. **Geração fundamentada:** a API da Cohere, usada no livro, recebe os documentos em um parâmetro próprio (`documents`). A API da NVIDIA não tem esse parâmetro, então os trechos são **numerados e inseridos no próprio prompt** (`[0]`, `[1]`, `[2]`).

O prompt tem duas mensagens:

* **sistema:** instrui o modelo a responder usando **somente** os documentos fornecidos, a citar o documento que apoia cada afirmação no formato `[n]` e a dizer quando os documentos não contêm a resposta;
* **usuário:** os documentos numerados, seguidos da pergunta.

A temperatura baixa (`0.2`) reduz a variação entre execuções, o que é desejável quando a resposta deve se limitar aos fatos dos documentos.

Experimente também a consulta comentada `"is it realistic?"` ("é realista?") e uma pergunta cuja resposta **não** está no texto, para verificar se o modelo admite que não sabe.

In [63]:
query = "income generated"
#query = "is it realistic?"

# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
results = search(query)

# 2- Grounded Generation
# Sem o parâmetro "documents" da Cohere, os trechos são numerados no prompt
docs = list(results['texts'])
numbered_docs = "\n".join(f"[{i}] {doc}" for i, doc in enumerate(docs))

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system",
         "content": "Answer the question using only the documents provided. "
                    "After each claim, cite the supporting document as [n]. "
                    "If the documents do not contain the answer, say so."},
        {"role": "user", "content": f"Documents:\n{numbered_docs}\n\nQuestion: {query}"},
    ],
    temperature=0.2,
    max_tokens=500,
)

print(response.choices[0].message.content)

Query:'income generated'
Nearest neighbors:
The film generated over $677 million worldwide (and $773 million with subsequent re‑releases) [0].


Para entender o que foi enviado ao modelo, vamos inspecionar o trecho do prompt com os documentos numerados. Esse é o **contexto** que fundamenta a resposta: tudo o que o modelo deveria usar está aqui.

In [64]:
numbered_docs

'[0] The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014\n[1] It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight\n[2] It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'

A seguir, o objeto completo devolvido pela API. Repare em dois campos:

* `content`: a resposta final, com as citações `[n]`;
* `reasoning_content`: o **raciocínio interno** do modelo. O `CHAT_MODEL` usado aqui é um modelo de raciocínio (*reasoning*): antes de responder, ele gera uma linha de pensamento, como no Chain-of-Thought da aula de Engenharia de Prompt. Esses tokens também são gerados e contabilizados, por isso a resposta demora mais. Em tarefas simples, o raciocínio pode ser desligado com o parâmetro `extra_body={"chat_template_kwargs": {"enable_thinking": False}}`.

In [65]:
response

ChatCompletion(id='chatcmpl-484d92f5-f48d-4e70-8048-601c8821dcee', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The film generated over $677\u202fmillion worldwide (and $773\u202fmillion with subsequent re‑releases) [0].', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='We need to answer "income generated". Likely refers to worldwide gross. Document [0] says worldwide gross over $677 million (and $773 million with re-releases). So answer: over $677 million (or $773 million). The question: "income generated". Use only documents. So we can say the film generated over $677 million worldwide gross, [0].\n'))], created=1791322431, model='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=123, prompt_tokens=152, total_tokens=275, completion_token

Por fim, extraímos as citações. A expressão regular encontra os marcadores `[n]` na resposta e mostra o trecho correspondente, para que o usuário possa **verificar a fonte** de cada afirmação.

Lembre-se de que essas citações foram **pedidas no prompt**, e não garantidas pela API: o modelo pode citar o documento errado ou fazer uma afirmação sem citação. Em um sistema real, vale verificar automaticamente se cada número citado existe e, se possível, se o trecho citado de fato apoia a afirmação.

In [66]:
import re

# Citações pedidas no prompt: extrai os marcadores [n] e mostra o trecho correspondente
answer = response.choices[0].message.content
cited_ids = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
for n in cited_ids:
    if n < len(docs):
        print(f"[{n}] {docs[n]}")

[0] The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014


# Síntese

| Etapa | Ferramenta neste notebook | Ideia principal |
|---|---|---|
| Divisão em blocos | `text.split('.')` | o que vira um vetor define o que pode ser encontrado |
| Embeddings | `nemotron-3-embed-1b` | textos com significados parecidos ficam próximos; pergunta e documento são codificados de forma diferente |
| Índice | FAISS `IndexFlatL2` | busca exata pelo vizinho mais próximo; em grande escala, índices aproximados e bancos vetoriais |
| Busca lexical | BM25 | forte com palavras exatas, fraca com sinônimos e paráfrases |
| Reranking | `llama-nemotron-rerank-vl-1b-v2` | segunda etapa, mais precisa e mais cara, aplicada a poucos candidatos |
| RAG | `nemotron-3-nano-omni-30b-a3b-reasoning` | o LLM responde com base nos trechos recuperados e cita as fontes |

## Para explorar

1. **Perguntas em português.** O modelo de embeddings é multilíngue. Pergunte "Quanto o filme arrecadou?" ou "A ciência do filme era precisa?" e verifique se a busca encontra as frases em inglês.
2. **Divisão em blocos.** Troque a divisão por frase pela divisão por parágrafo (`text.split('\n\n')`) e repita a busca do item 4. Os resultados ficam melhores ou piores? Por quê?
3. **Similaridade de cosseno.** Crie um índice `faiss.IndexFlatIP`, normalize os vetores com `faiss.normalize_L2` (antes de adicioná-los e antes de cada busca) e compare a ordem dos resultados com a do `IndexFlatL2`.
4. **Pergunta sem resposta no RAG.** Faça ao sistema RAG a pergunta sobre a massa da Lua. O modelo admite que os documentos não contêm a resposta?
5. **Busca híbrida.** Combine os resultados da busca densa e do BM25 em uma única lista de candidatos e use o reranker para ordená-la.

## Referências

ALAMMAR, J.; GROOTENDORST, M. *Hands-On Large Language Models: Language Understanding and Generation*. Sebastopol: O'Reilly Media, 2024. Cap. 8: Semantic Search and Retrieval-Augmented Generation.

JOHNSON, J.; DOUZE, M.; JÉGOU, H. Billion-scale similarity search with GPUs. *IEEE Transactions on Big Data*, v. 7, n. 3, p. 535-547, 2021. (FAISS)

LEWIS, P. et al. Retrieval-augmented generation for knowledge-intensive NLP tasks. *Advances in Neural Information Processing Systems*, v. 33, p. 9459-9474, 2020. Disponível em: https://arxiv.org/abs/2005.11401

NOGUEIRA, R.; CHO, K. Passage re-ranking with BERT. *arXiv preprint* arXiv:1901.04085, 2019.

ROBERTSON, S.; ZARAGOZA, H. The probabilistic relevance framework: BM25 and beyond. *Foundations and Trends in Information Retrieval*, v. 3, n. 4, p. 333-389, 2009.

NVIDIA. Catálogo de modelos (build.nvidia.com). Disponível em: https://build.nvidia.com/models